# NeoKYC — F³-Net GPU Training & Evaluation
**Paper**: *Thinking in Frequency: Face Forgery Detection by Mining Frequency-aware Clues* (Qian et al., ECCV 2020)

This standalone Google Colab / Kaggle notebook trains the complete dual-stream F³-Net model with ImageNet-pretrained Xception backbones and MixBlock collaborative cross-attention on GPU (T4 / V100 / A100).

In [ ]:
# [1] Verify GPU availability
!nvidia-smi

In [ ]:
# [2] Clone NeoKYC repository or upload codebase
# If cloning from GitHub:
# !git clone https://github.com/PranaySolanki/NeoKYC.git
# %cd NeoKYC

!pip install -q torch torchvision opencv-python-headless pillow tqdm scikit-learn scipy

In [ ]:
# [3] Download Pretrained ImageNet Xception Weights
!mkdir -p checkpoints/pretrained
!curl -L -o checkpoints/pretrained/xception-b5690688.pth \
  https://huggingface.co/spaces/asdasdasdasd/Face-forgery-detection/resolve/main/xception-b5690688.pth

# Verify file size (expected: 91,674,713 bytes)
!ls -lh checkpoints/pretrained/xception-b5690688.pth

In [ ]:
# [4] Dataset Setup: FaceForensics++ (c40 compression)
# If using Kaggle, attach the FaceForensics++ dataset.
# If in Colab, symlink your Google Drive dataset folder:
# from google.colab import drive
# drive.mount('/content/drive')
# !ln -s /content/drive/MyDrive/FaceForensics++ data/FaceForensics++

# Alternatively, for pipeline testing, generate verification data:
!python -m backend.data.prepare_subset --target_dir data/FaceForensics++ --train_vids 20 --val_vids 10 --test_vids 10 --frames 10

In [ ]:
# [5] Train Dual-Stream F³-Net with Pretrained Backbone on GPU
!python backend/train.py \
  --dataset_dir data/FaceForensics++ \
  --output_dir checkpoints/f3net \
  --pretrained_xception checkpoints/pretrained/xception-b5690688.pth \
  --mode Mix \
  --compression c40 \
  --batch_size 16 \
  --epochs 10 \
  --lr 0.002

In [ ]:
# [6] Run Comprehensive Evaluation & Calibrate Decision Threshold
!python -m backend.evaluation.evaluator \
  --checkpoint checkpoints/f3net/best_model.pth \
  --data_root data/FaceForensics++ \
  --split test \
  --compression c40 \
  --calibrate_on val \
  --output evaluation_results.json

In [ ]:
# [7] Display Evaluation Results
import json
with open('evaluation_results.json') as f:
    res = json.load(f)
print(json.dumps(res, indent=2))

# Download checkpoints/f3net/best_model.pth to your local machine at NeoKYC/checkpoints/f3net/best_model.pth